<a href="https://colab.research.google.com/github/nmemranhussain/DiaVigil/blob/main/notebooks/03_diavigil_nested_cv_and_probability_calibration.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

**Cell 1: Environment Setup & Core Imports**

In [1]:
from google.colab import drive
import os
import joblib
import pandas as pd
import numpy as np

drive.mount('/content/drive')

BASE_DIR = '/content/drive/MyDrive/DiaVigil'
DATA_PROCESSED = os.path.join(BASE_DIR, 'data/processed')

# Load Development Cohort only
dev_df = pd.read_parquet(os.path.join(DATA_PROCESSED, 'dev_cohort.parquet'))
pipeline_path = os.path.join(DATA_PROCESSED, 'feature_engineering_pipeline.joblib')

print(f"Dev Cohort loaded: {len(dev_df):,} encounters")
print(f"Base target rate: {dev_df['target'].mean() * 100:.2f}%")

Mounted at /content/drive
Dev Cohort loaded: 80,239 encounters
Base target rate: 11.36%


**Cell 2: Install Advanced Evaluation & Modeling Dependencies**

In [2]:
!pip install -q catboost optuna netcal
import catboost
import optuna
import netcal
print("Dependencies verified: CatBoost, Optuna, and Netcal installed.")

Dependencies verified: CatBoost, Optuna, and Netcal installed.


**Cell 3: Patient-Group Cross-Validation Splitter**

Set up an internal 5-fold StratifiedGroupKFold split within the development cohort so that training, probability calibration, and threshold selection never evaluate on overlapping patients.

In [3]:
from sklearn.model_selection import StratifiedGroupKFold

# Separate features from target and grouping metadata
X_dev = dev_df.drop(columns=['target', 'encounter_id'])
y_dev = dev_df['target'].values
groups_dev = dev_df['patient_nbr'].values

# Inner CV scheme
inner_sgkf = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=42)

print(f"Internal CV generator configured: 5 folds grouped on patient_nbr.")

Internal CV generator configured: 5 folds grouped on patient_nbr.


**Cell 4: Base Model Training & Isotonic Probability Calibration**

Train CatBoost on fold training sets and calibrate output probabilities using Isotonic Regression on held-out validation folds.

In [4]:
import re
import joblib
import numpy as np
import pandas as pd
from sklearn.base import BaseEstimator, TransformerMixin
from catboost import CatBoostClassifier
from sklearn.calibration import CalibratedClassifierCV
from sklearn.metrics import average_precision_score, brier_score_loss

# 1. Define custom classes in __main__ so joblib can deserialize the pipeline
ELIXHAUSER_ICD9_RULES = {
    'elix_congestive_heart_failure': r'^(39891|40201|40211|40291|40401|40403|40411|40413|40491|40493|428)',
    'elix_cardiac_arrhythmias': r'^(42610|42611|42613|4262|4263|4264|42651|42652|42653|4266|4267|4268|4270|4272|4273|4276|4278|4279|7850|V450|V533)',
    'elix_valvular_disease': r'^(0932|394|395|396|397|424|7463|7464|7465|7466|V422|V433)',
    'elix_pulmonary_circulation': r'^(416|4179)',
    'elix_peripheral_vascular': r'^(440|441|442|443|444|4471|5571|5579|V434)',
    'elix_hypertension_uncomplicated': r'^(401)',
    'elix_hypertension_complicated': r'^(402|403|404|405)',
    'elix_diabetes_complicated': r'^(250[4-9])',
    'elix_renal_failure': r'^(585|586|V420|V451|V56)',
    'elix_fluid_electrolyte': r'^(276)',
    'elix_chronic_pulmonary': r'^(490|491|492|493|494|495|496|500|501|502|503|504|505)',
    'elix_coagulopathy': r'^(286|2871|2873|2874|2875)'
}

def extract_elixhauser_flags(df: pd.DataFrame) -> pd.DataFrame:
    diag_cols = ['diag_1', 'diag_2', 'diag_3']
    comorb_features = pd.DataFrame(index=df.index)
    cleaned_diags = df[diag_cols].fillna('').astype(str).apply(
        lambda col: col.str.strip().str.replace('.', '', regex=False)
    )
    for flag_name, pattern in ELIXHAUSER_ICD9_RULES.items():
        compiled_regex = re.compile(pattern)
        matches = cleaned_diags.apply(
            lambda row: any(bool(compiled_regex.match(val)) for val in row if val),
            axis=1
        )
        comorb_features[flag_name] = matches.astype(int)
    return comorb_features

class ClinicalDomainFeatureEngineer(BaseEstimator, TransformerMixin):
    def __init__(self):
        self.medication_columns = [
            'metformin', 'repaglinide', 'nateglinide', 'chlorpropamide',
            'glimepiride', 'acetohexamide', 'glipizide', 'glyburide', 'tolbutamide',
            'pioglitazone', 'rosiglitazone', 'acarbose', 'miglitol', 'troglitazone',
            'tolazamide', 'examide', 'citoglipton', 'insulin', 'glyburide-metformin',
            'glipizide-metformin', 'glimepiride-pioglitazone', 'metformin-rosiglitazone',
            'metformin-pioglitazone'
        ]

    def fit(self, X, y=None):
        return self

    def transform(self, X):
        X_out = X.copy()
        comorb_df = extract_elixhauser_flags(X_out)
        for col in comorb_df.columns:
            X_out[col] = comorb_df[col].values

        X_out['total_elixhauser_burden'] = comorb_df.sum(axis=1).values
        X_out['healthcare_utilization_index'] = (
            X_out['number_outpatient'].astype(float) +
            (2.0 * X_out['number_emergency'].astype(float)) +
            (3.0 * X_out['number_inpatient'].astype(float))
        )
        active_meds = X_out[self.medication_columns].apply(lambda row: (row != 'No').sum(), axis=1)
        med_titrations = X_out[self.medication_columns].apply(lambda row: row.isin(['Up', 'Down']).sum(), axis=1)

        X_out['active_diabetes_med_count'] = active_meds
        X_out['dosage_titration_count'] = med_titrations
        X_out['on_insulin'] = (X_out['insulin'] != 'No').astype(int)
        X_out['encounter_severity_index'] = (
            X_out['time_in_hospital'].astype(float) *
            (X_out['num_medications'].astype(float) / (X_out['num_lab_procedures'].astype(float) + 1.0))
        )
        X_out['glycemic_status_code'] = X_out['A1Cresult'].map({
            'None': 0, 'Norm': 1, '>7': 2, '>8': 2
        }).fillna(0).astype(int)
        return X_out

# 2. Extract split fold
train_fold_idx, val_fold_idx = next(inner_sgkf.split(X_dev, y_dev, groups=groups_dev))
train_data = dev_df.iloc[train_fold_idx].copy()
val_data = dev_df.iloc[val_fold_idx].copy()

print(f"Train split size: {len(train_data):,} | Validation split size: {len(val_data):,}")

# 3. Load feature pipeline and transform
pipeline = joblib.load(pipeline_path)
print("Loading feature pipeline...")
X_train_trans = pipeline.fit_transform(train_data.drop(columns=['target', 'encounter_id', 'patient_nbr']))
y_train_fold = train_data['target'].values

X_val_trans = pipeline.transform(val_data.drop(columns=['target', 'encounter_id', 'patient_nbr']))
y_val_fold = val_data['target'].values
print(f"Transformed train shape: {X_train_trans.shape} | Val shape: {X_val_trans.shape}")

# 4. Train CatBoost with visible progress logging
print("Training CatBoost Classifier...")
cat_model = CatBoostClassifier(
    iterations=400,
    learning_rate=0.05,
    depth=6,
    auto_class_weights='Balanced',
    random_seed=42,
    verbose=100
)
cat_model.fit(X_train_trans, y_train_fold)

# 5. Fit Isotonic Calibrator
print("Fitting Isotonic Probability Calibrator...")
try:
    calibrator = CalibratedClassifierCV(estimator=cat_model, method='isotonic', cv='prefit')
    calibrator.fit(X_val_trans, y_val_fold)
except (ValueError, TypeError):
    from sklearn.frozen import FrozenEstimator
    calibrator = CalibratedClassifierCV(estimator=FrozenEstimator(cat_model), method='isotonic')
    calibrator.fit(X_val_trans, y_val_fold)

raw_val_probs = cat_model.predict_proba(X_val_trans)[:, 1]
cal_val_probs = calibrator.predict_proba(X_val_trans)[:, 1]

print("--- Fold Validation Diagnostics ---")
print(f"Raw PR-AUC:       {average_precision_score(y_val_fold, raw_val_probs):.4f}")
print(f"Raw Brier Score:  {brier_score_loss(y_val_fold, raw_val_probs):.4f}")
print(f"Calibrated Brier: {brier_score_loss(y_val_fold, cal_val_probs):.4f}")

Train split size: 64,208 | Validation split size: 16,031
Loading feature pipeline...
Transformed train shape: (64208, 98) | Val shape: (16031, 98)
Training CatBoost Classifier...
0:	learn: 0.6896384	total: 70ms	remaining: 27.9s
100:	learn: 0.6394051	total: 2.04s	remaining: 6.03s
200:	learn: 0.6287325	total: 3.96s	remaining: 3.92s
300:	learn: 0.6147959	total: 5.89s	remaining: 1.94s
399:	learn: 0.6022465	total: 7.82s	remaining: 0us
Fitting Isotonic Probability Calibrator...
--- Fold Validation Diagnostics ---
Raw PR-AUC:       0.2349
Raw Brier Score:  0.2121
Calibrated Brier: 0.0961


/usr/local/lib/python3.13/dist-packages/sklearn/calibration.py:333: UserWarning: The `cv='prefit'` option is deprecated in 1.6 and will be removed in 1.8. You can use CalibratedClassifierCV(FrozenEstimator(estimator)) instead.
  warnings.warn(


**Cell 5: Expected Calibration Error (ECE) & Reliability Analysis**

Quantify pre- and post-calibration error across 10 probability bins to confirm risk predictions match observed readmission rates.

In [5]:
def calculate_ece(y_true, probs, n_bins=10):
    bin_edges = np.linspace(0.0, 1.0, n_bins + 1)
    ece = 0.0
    for i in range(n_bins):
        bin_mask = (probs >= bin_edges[i]) & (probs < bin_edges[i+1])
        bin_size = np.sum(bin_mask)
        if bin_size > 0:
            bin_acc = np.mean(y_true[bin_mask])
            bin_conf = np.mean(probs[bin_mask])
            ece += (bin_size / len(y_true)) * np.abs(bin_acc - bin_conf)
    return ece

ece_raw = calculate_ece(y_val_fold, raw_val_probs)
ece_cal = calculate_ece(y_val_fold, cal_val_probs)

print(f"Uncalibrated ECE: {ece_raw:.4f}")
print(f"Calibrated ECE:   {ece_cal:.4f}")
print(f"ECE Reduction:    {((ece_raw - ece_cal) / ece_raw) * 100:.2f}%")

Uncalibrated ECE: 0.3289
Calibrated ECE:   0.0000
ECE Reduction:    100.00%


**Cell 6: Cost-Utility Threshold Optimization**

Determine the operating threshold $\tau^*$ by minimizing asymmetric clinical misclassification cost ($C_{FN} = 5 \times C_{FP}$) strictly on validation fold probabilities.Python

In [6]:
import numpy as np

threshold_grid = np.linspace(0.02, 0.40, 100)
c_fp = 1.0   # Cost of an unnecessary care management alert
c_fn = 5.0   # Cost of an unflagged, missed 30-day readmission

total_costs = []
for t in threshold_grid:
    fp = np.sum((cal_val_probs >= t) & (y_val_fold == 0))
    fn = np.sum((cal_val_probs < t) & (y_val_fold == 1))
    cost = (c_fp * fp) + (c_fn * fn)
    total_costs.append(cost)

optimal_threshold = threshold_grid[np.argmin(total_costs)]
print(f"Optimal Decision Cutoff (tau*): {optimal_threshold:.4f}")
print(f"Minimum Clinical Misclassification Cost: {np.min(total_costs):,}")

# Persist trained model, calibrator, and optimal cutoff
model_artifacts = {
    'fitted_pipeline': pipeline,
    'champion_model': cat_model,
    'calibrator': calibrator,
    'optimal_threshold': optimal_threshold
}
artifact_path = os.path.join(DATA_PROCESSED, 'model_calibration_artifacts.joblib')
joblib.dump(model_artifacts, artifact_path)
print(f"Model artifacts persisted to: {artifact_path}")

Optimal Decision Cutoff (tau*): 0.1620
Minimum Clinical Misclassification Cost: 8,078.0
Model artifacts persisted to: /content/drive/MyDrive/DiaVigil/data/processed/model_calibration_artifacts.joblib
